# 숫자 이미지를 이용한 숫자 판별 예제

## 준비

In [2]:
import os
import numpy as np
from sklearn.datasets import load_digits
from PIL import Image

# 이미지 불러오기
digits = load_digits()
print(f"데이터 shape : {digits.data.shape}") # 8 x 8 이미지를 일렬로 나열
print(f"이미지 shape : {digits.images.shape}") # 8px X 8px 숫자 이미지

# 이미지 저장하기
# 1. 폴더 생성
OUT_DIR = "digits_images"
os.makedirs(OUT_DIR, exist_ok=True)
# 0~9까지 있음. 폴더를 0~9
for i in range(10):
    os.makedirs(f"{OUT_DIR}/{i}", exist_ok=True)

데이터 shape : (1797, 64)
이미지 shape : (1797, 8, 8)


In [3]:
# 2.이미지 저장
# enumerate는 index와 value를 줌
def image_save():
    for i, (img, label) in enumerate(zip(digits.images, digits.target)):
        arr = (img / 16 * 255).astype(np.uint8)
        # 숫자가 0이면 0폴더에 0_0001.png형태로 저장
        Image.fromarray(arr, mode="L").save(f"{OUT_DIR}/{label}/{label}_{i:04d}.png")

# 1. 데이터 선택

In [5]:
X = digits.data
y = digits.target 
# 1797장의 숫자 이미지정보. 이미지를 한줄로 나열
print(f"X: {X.shape}, y:{y.shape}")

X: (1797, 64), y:(1797,)


# 2. 사전 준비

In [8]:
# 저장된 값이 밝기 
# 범위가 0~16
# 범위를 0~1로 바꿈
X = X / 16

# 3.훈련, 테스트 분리

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. 학습

In [17]:
from sklearn.neural_network import MLPClassifier # 분류
# verbose=True : 에포크마다 몇번째 작업인지 손실률은 얼마인지를 알려줌 
model = MLPClassifier(hidden_layer_sizes=(64, ), max_iter=500, random_state=42)
model.fit(X_train, y_train)
print("학습완료")

학습완료


# 5.평가

In [19]:
from sklearn.metrics import accuracy_score
print(f"학습 정확도 : {accuracy_score(y_train, model.predict(X_train)):.1f}")
print(f"시험 정확도 : {accuracy_score(y_test, model.predict(X_test)):.1f}")

학습 정확도 : 1.0
시험 정확도 : 1.0


# 사용자가 테스트할 이미지를 가져와서 데이터 형식으로 변환하기

In [21]:
import numpy as np
from PIL import ImageFilter, ImageOps

def load_my_image(path):
    """내 그림 → 학습 데이터와 같은 모양 (숫자 64개, 0~1)으로 바꾼다"""
    image = Image.open(path).convert("RGBA")
    background = Image.new("RGBA", image.size, "white")   # 투명 배경 PNG는 흰 바탕으로 채운다
    image = Image.alpha_composite(background, image)
 
    image = image.convert("L")              # 1) 흑백으로 (0: 검정 ~ 255: 흰색)
 
    # 2) 글씨가 큰 값이 되게 맞춘다 (학습 데이터와 같게)
    #    흰 바탕에 검은 글씨면 뒤집고, 검은 바탕에 흰 글씨면 그대로 둔다
    #    바탕이 그림 대부분을 차지하니까, 평균 밝기가 밝으면 흰 바탕이다
    if np.array(image).mean() > 127:
        image = ImageOps.invert(image)
 
    # 3) 글씨가 있는 부분만 잘라 낸다 (학습 데이터의 숫자는 칸을 꽉 채운다)
    box = image.point(lambda p: 255 if p > 50 else 0).getbbox()
    if box is None:
        raise ValueError("그림에서 글씨를 찾지 못했습니다. 흰 바탕에 검은색으로 그렸는지 확인하세요")
    image = image.crop(box)
 
    # 4) 정사각형으로 만들고 가장자리에 여백을 조금 준다 (가로세로 비율이 찌그러지지 않게)
    size = max(image.size)
    square = Image.new("L", (size, size), 0)
    square.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    square = ImageOps.expand(square, border=size // 8, fill=0)
 
    # 5) 글씨를 두껍게 한다. 가는 선은 8 x 8로 줄이면 흐려져서 사라진다
    #    (학습 데이터의 숫자는 8 x 8에서 선 굵기가 1~2칸이다)
    square = square.filter(ImageFilter.MaxFilter(max(3, (size // 12) | 1)))
 
    # 6) 8 x 8로 줄인다. BOX: 칸마다 평균을 낸다 (학습 데이터를 만든 방식과 비슷하다)
    small = square.resize((8, 8), Image.Resampling.BOX)
 
    # 7) 밝기 0~255 → 0~16 → 0~1 (학습 때 X / 16 과 같게)
    pixels = np.array(small, dtype=float) / 255 * 16
    return pixels.reshape(1, 64) / 16, Image.open(path)

In [22]:
X[0]

array([0.    , 0.    , 0.3125, 0.8125, 0.5625, 0.0625, 0.    , 0.    ,
       0.    , 0.    , 0.8125, 0.9375, 0.625 , 0.9375, 0.3125, 0.    ,
       0.    , 0.1875, 0.9375, 0.125 , 0.    , 0.6875, 0.5   , 0.    ,
       0.    , 0.25  , 0.75  , 0.    , 0.    , 0.5   , 0.5   , 0.    ,
       0.    , 0.3125, 0.5   , 0.    , 0.    , 0.5625, 0.5   , 0.    ,
       0.    , 0.25  , 0.6875, 0.    , 0.0625, 0.75  , 0.4375, 0.    ,
       0.    , 0.125 , 0.875 , 0.3125, 0.625 , 0.75  , 0.    , 0.    ,
       0.    , 0.    , 0.375 , 0.8125, 0.625 , 0.    , 0.    , 0.    ])